<a href="https://colab.research.google.com/github/chhavi-s9/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 Research Question: CTR / Engagement Opportunity Scoring

> **Status:** Provisional lane selection  
> **Dataset:** FlyRank starter dataset  
> **Purpose:** Decide whether CTR / engagement opportunity scoring is a useful problem to investigate over the next seven weeks.

## Executive summary

**Lane:** Lane 4, CTR / Engagement Opportunity Scoring

**Research question:** Which visible pages appear to under-capture clicks or engagement relative to comparable pages and therefore deserve metadata, content, or monitoring review?

**Decision:** Which pages should a review team investigate first?

**Important framing:** This is a prioritisation problem, not a claim that low CTR proves poor content or metadata. The analysis will account for visibility, position, and volume before treating a page as a review candidate.


## 1. My lane and why

I am choosing **Lane 4: CTR / Engagement Opportunity Scoring**.

The lane is useful because it connects observed search performance to a concrete content-review decision. A page can receive meaningful exposure but capture fewer clicks or weaker engagement than comparable pages. A review queue could help a team focus limited optimisation capacity on pages with the strongest evidence of an opportunity.

The problem is intentionally framed as **opportunity identification**, not diagnosis. Low CTR alone does not prove that a title, meta description, content quality, or search-intent match is the cause. Position and impression volume can strongly affect observed CTR, so pages should be compared with appropriate visibility controls.

The eventual goal would be a transparent ranked list of candidates with reason codes that a human reviewer can understand and act on.


## 2. The question: decision, action, and risk

### Search question

**Which visible pages appear to under-capture clicks or engagement relative to comparable pages and therefore deserve metadata, content, or monitoring review?**

### Unit of analysis

For the starter dataset, the working unit is the content/page observation represented by each row. Before any future warehouse analysis, the grain will be checked explicitly because the daily performance table is at a finer time-based grain.

### Output

A ranked list of review candidates, with interpretable reason codes such as:

- meaningful impression volume;
- strong or useful search visibility;
- CTR below the expected level for comparable position observations;
- weak post-click engagement;
- sufficient evidence to justify human review.

### Decision

**Which pages should the review team investigate first?**

### Action

A reviewer could inspect the selected candidates for metadata, content/intent alignment, snippet structure, on-page engagement, or continued monitoring.

### Cost of a wrong recommendation

**False positive:** limited review capacity is spent on a page with little improvement opportunity.

**False negative:** a page with meaningful visibility and a potentially useful improvement opportunity is missed.

The cost therefore depends on both the value of the pages being reviewed and the team's review capacity. This is why a ranked queue and top-K validation are more relevant than generic classification accuracy.

### Why data or ML can help

Manual review does not scale well when there are many content items. The starter data contains multiple signals that can be considered together, including impressions, clicks/CTR, average position, position tier, and engagement measures. A transparent statistical comparison can therefore narrow a large population into a smaller review queue. ML may become useful later only if it improves that prioritisation beyond a simple, interpretable baseline.


## 3. Quick look at the data

The starter notebook reports **30,000 rows and 44 columns**. The dataset includes Lane 4 relevant fields such as `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `impression_tier`, and `position_tier`.

**The remaining evidence below is calculated directly from the local starter CSV when this notebook is executed. No statistics are hard-coded.**


In [8]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

lane4_fields = [
    "impressions", "clicks", "ctr", "avg_position",
    "position_tier", "impression_tier", "engagement_rate", "scroll_rate"
]
print("\nLane 4 fields present:")
print([c for c in lane4_fields if c in df.columns])

Dataset shape: 30,000 rows × 44 columns

Lane 4 fields present:
['ctr', 'avg_position', 'position_tier', 'impression_tier', 'engagement_rate', 'scroll_rate']


In [9]:
# This cell is redundant as the data has already been loaded in the previous cell (0789fa26).

In [10]:
# Evidence 1: overall CTR, visibility, and engagement summary
summary_cols = [c for c in [
    "ctr", "avg_position", "engagement_rate", "scroll_rate"
] if c in df.columns]

summary = df[summary_cols].describe().T.round(3)
summary

,count,mean,std,min,25%,50%,75%,max
ctr,30000.0,0.511,3.279,0.0,0.0,0.07,0.29,100.0
avg_position,30000.0,16.342,15.217,0.0,6.2,10.80,22.30,245.0
engagement_rate,30000.0,2.535,8.310,0.0,0.0,0.00,1.35,100.0
scroll_rate,29875.0,18.213,29.473,0.0,0.0,5.00,23.53,300.0


In [11]:
# Evidence 2: how observations are distributed across impression tiers
if "impression_tier" in df.columns:
    impression_summary = (
        df["impression_tier"]
        .value_counts(dropna=False)
        .rename_axis("impression_tier")
        .reset_index(name="observations")
    )
    impression_summary["share"] = (
        impression_summary["observations"] / len(df)
    ).round(3)
    display(impression_summary)
else:
    print("impression_tier is not present in this version of the starter data.")

,impression_tier,observations,share
0,low,11248,0.375
1,moderate,10469,0.349
2,good,7205,0.240
3,excellent,1078,0.036


In [12]:
# Evidence 3: CTR variation within position tiers
if {"position_tier", "ctr"}.issubset(df.columns):
    tier_summary = (
        df.groupby("position_tier")["ctr"]
        .agg(
            observations="count",
            median_ctr="median",
            mean_ctr="mean",
            p25_ctr=lambda x: x.quantile(0.25),
            p75_ctr=lambda x: x.quantile(0.75),
        )
        .round(4)
        .sort_index()
    )
    display(tier_summary)
else:
    print("Required position/CTR fields are not present in this version of the starter data.")

,observations,median_ctr,mean_ctr,p25_ctr,p75_ctr
position_tier,,,,,
deep,1319,0.00,0.1502,0.0,0.00
page_1,11814,0.16,0.6525,0.0,0.41
page_3_5,7242,0.03,0.2225,0.0,0.17
striking,7304,0.11,0.3232,0.0,0.30
top_3,2321,0.00,1.4836,0.0,0.00


### How these numbers support the lane

The evidence should establish three things:

1. **Scale:** there are enough observations to investigate a prioritisation problem.
2. **Opportunity/volume:** a meaningful portion of observations has enough visibility to make CTR or engagement worth considering.
3. **Variation:** CTR differs across observations even when position is considered, leaving a useful question about which pages under-capture relative to comparable pages.

The exact statistics above are generated from the starter data rather than copied from an external example. Their interpretation should remain descriptive and should not be treated as causal evidence.


## 4. Initial analytical direction

The first baseline should be simple and explainable rather than immediately jumping to a complex model.

### Candidate approach

1. Define comparable visibility groups using position or position tiers.
2. Estimate expected CTR within those groups.
3. Measure the gap between observed CTR and the comparable expectation.
4. Apply a minimum-volume policy so very small samples do not dominate the queue.
5. Combine the gap with evidence of opportunity, such as impressions or sessions, to produce a ranked review list.
6. Add engagement signals where the decision is specifically about post-click behaviour.

A later model should have to beat this transparent baseline. The model is not the objective; improving the review decision is.


## 5. Thresholds are policy choices

No single impression count or CTR gap should be treated as a universal definition of an opportunity.

Potential policies include:

- minimum impressions required for review;
- minimum size of the below-expected CTR gap;
- maximum number of candidates a team can review;
- confidence or evidence levels for different actions.

The chosen threshold should be tested by examining how many observations it captures and how the resulting top-K queue changes. The final choice should reflect the real review capacity and the relative cost of false positives versus false negatives.

For a ranked queue, metrics such as **Precision@20** or **Precision@50** are likely to be more decision-aligned than generic accuracy if the team can realistically review 20 or 50 candidates.


## 6. Leakage and feature-policy check

Precalculated fields such as tiers or existing product outputs require explicit treatment.

Before using any such field, ask:

1. What is it trying to measure?
2. Was it available at the decision time?
3. Is it context, a feature, a label/proxy, or an output?
4. Could it leak the answer?
5. Does the result still hold if the field is removed?

**Do not use an existing `priority_score`, `action_type`, `health_score`, or decision flag as an ordinary feature if it represents the product decision we are trying to rebuild.**

The goal is to learn from underlying observable signals, not to reproduce an existing product output by feeding it back into a model.


## 7. Careful words: what I can and cannot claim

### I can claim

- The starter data contains observed CTR, visibility, and engagement signals.
- CTR can be compared within comparable position groups rather than across all pages indiscriminately.
- Some observations may have meaningful visibility but comparatively weak observed CTR or engagement.
- These patterns can be used to prioritise candidates for human review.

### I cannot claim from this dataset alone

- Low CTR proves that a title or meta description is bad.
- A content refresh caused a recovery, unless an explicit experiment or causal design supports that conclusion.
- The dataset reveals Google's ranking algorithm or its causal ranking factors.
- The dataset proves anything about AI citations or AI rankings.
- A ranked candidate is guaranteed to improve after intervention.

The final output should therefore use language such as **observed**, **associated with**, **suggests**, **may warrant review**, and **candidate**, rather than causal or guaranteed language.


## 8. Public-safe output

The eventual public research paper must not expose raw private identifiers or examples that could reveal a client.

**Allowed:** pseudonymised IDs, aggregated metrics, safe charts, high-level examples, and generic content actions.

**Not allowed:** client names, domains, URLs from the data, raw private queries, titles or text that could identify a client, credentials, BigQuery internals, or unsupported claims about Google.

This Week 1 notebook therefore focuses on aggregate evidence and methodological framing rather than publishing individual page examples.


## 9. Why this is worth the next seven weeks

The problem has a clear decision, observable inputs, a measurable ranking objective, and a realistic human action. It also contains methodological challenges that are meaningful for applied ML: position adjustment, low-volume noise, threshold selection, leakage prevention, ranking evaluation, and validation against a transparent baseline.

The lane is therefore worth investigating provisionally. The decision can still be changed through Week 4 if later analysis shows that the starter or warehouse data does not support a useful opportunity ranking.


## 10. Self-check

- [x] One predefined lane selected.
- [x] Search question stated.
- [x] Unit of analysis identified provisionally and grain will be rechecked before warehouse modelling.
- [x] Decision named.
- [x] Output named.
- [x] Action named.
- [x] Cost of a wrong recommendation explained.
- [x] Why data/ML may help explained without assuming ML is necessary.
- [x] At least two data-driven evidence cells included.
- [x] Position is treated as an important comparison context.
- [x] Low-volume noise is acknowledged.
- [x] Thresholds are treated as policy choices.
- [x] Leakage risks are explicitly addressed.
- [x] Causal claims are explicitly avoided.
- [x] Public-safe output rules are documented.
- [x] Execute all code cells and inspect the generated statistics.
- [x] Commit the executed notebook to `work/notebooks/w01_research_question.ipynb`.


## Sources

1. FlyRank ML Internship Starter repository: https://github.com/chhavi-s9/flyrank-ml-internship-starter
2. FlyRank Lane Guide: `docs/ml-intern-dataset-and-lane-guide.md`
3. FlyRank starter discovery notebook: `notebooks/01_first_look_and_discovery.ipynb`
4. FlyRank full-release notebook: `notebooks/03_working_with_the_full_release.ipynb`
5. FlyRank warehouse release: https://huggingface.co/datasets/FlyRank/internship-warehouse
6. Google Search Central, Search performance data: https://developers.google.com/search/blog/2022/10/performance-data-deep-dive
7. Google Search Central, Debugging Search traffic drops: https://developers.google.com/search/docs/monitor-debug/debugging-search-traffic-drops

**Reproducibility note:** The key evidence cells load the local starter CSV and calculate the statistics at execution time. No dataset statistics are fabricated or hard-coded.
